In [22]:
import json
from urllib.request import urlopen

url = "https://xkcd.com/info.0.json"

with urlopen(url) as response:
    data = json.load(response)

print(data)

{'month': '4', 'num': 3233, 'link': '', 'year': '2026', 'news': '', 'safe_title': 'Make It Myself', 'transcript': '', 'alt': "It's not as big a loss as it looks, because now I have have leftover supplies, which will help me talk myself into doing this all over again with a new project!", 'img': 'https://imgs.xkcd.com/comics/make_it_myself.png', 'title': 'Make It Myself', 'day': '15'}


In [23]:
print("Comic ID:", data["num"])
print("Title:", data["title"])
print("Safe title:", data["safe_title"])
print("Image URL:", data["img"])
print("Published date parts:", data["year"], data["month"], data["day"])

Comic ID: 3233
Title: Make It Myself
Safe title: Make It Myself
Image URL: https://imgs.xkcd.com/comics/make_it_myself.png
Published date parts: 2026 4 15


In [24]:
from google.cloud import bigquery
from datetime import datetime, UTC

client = bigquery.Client()

table_id = "xkcd-case-study-493421.xkcd_raw.raw_xkcd_comics"

row = {
    "comic_id": data["num"],
    "month": int(data["month"]),
    "day": int(data["day"]),
    "year": int(data["year"]),
    "title": data["title"],
    "safe_title": data["safe_title"],
    "alt": data["alt"],
    "img_url": data["img"],
    "transcript": data["transcript"],
    "news": data["news"],
    "link": data["link"],
    "published_date": f'{data["year"]}-{int(data["month"]):02d}-{int(data["day"]):02d}',
    "fetched_at": datetime.now(UTC).isoformat(),
    "source_url": "https://xkcd.com/info.0.json",
    "raw_json": str(data)
}

errors = client.insert_rows_json(table_id, [row])

if errors:
    print("Insert failed:", errors)
else:
    print("1 row inserted successfully")

1 row inserted successfully


In [25]:
import json
from urllib.request import urlopen
from google.cloud import bigquery
from datetime import datetime, UTC

client = bigquery.Client()
table_id = "xkcd-case-study-493421.xkcd_raw.raw_xkcd_comics"

# get latest comic number
with urlopen("https://xkcd.com/info.0.json") as response:
    latest_data = json.load(response)

latest_comic_id = latest_data["num"]
print("Latest comic id:", latest_comic_id)

# get already loaded comic ids from BigQuery
query = f"""
SELECT comic_id
FROM `{table_id}`
"""
existing_ids = {row.comic_id for row in client.query(query).result()}

print("Already loaded rows:", len(existing_ids))

rows_to_load = []

for comic_id in range(1, latest_comic_id + 1):
    if comic_id in existing_ids:
        continue

    url = f"https://xkcd.com/{comic_id}/info.0.json"

    try:
        with urlopen(url) as response:
            data = json.load(response)

        row = {
            "comic_id": data["num"],
            "month": int(data["month"]),
            "day": int(data["day"]),
            "year": int(data["year"]),
            "title": data["title"],
            "safe_title": data["safe_title"],
            "alt": data["alt"],
            "img_url": data["img"],
            "transcript": data["transcript"],
            "news": data["news"],
            "link": data["link"],
            "published_date": f'{data["year"]}-{int(data["month"]):02d}-{int(data["day"]):02d}',
            "fetched_at": datetime.now(UTC).isoformat(),
            "source_url": url,
            "raw_json": json.dumps(data)
        }

        rows_to_load.append(row)

    except Exception as e:
        print(f"Skipping comic {comic_id}: {e}")

print("Rows prepared for load:", len(rows_to_load))

Latest comic id: 3233
Already loaded rows: 1
Skipping comic 404: HTTP Error 404: Not Found
Rows prepared for load: 3231


In [26]:
job = client.load_table_from_json(rows_to_load, table_id)
job.result()

print("Full load completed successfully")
print("Rows loaded:", len(rows_to_load))

Full load completed successfully
Rows loaded: 3231
